# Robustness Analysis

Evaluate model behavior under input perturbations.

In [41]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score)


In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_CLASSES = 2

CLASS_NAMES = ["HP", "SSA"]

CONDITIONS = ["clean","brightness","gaussian_noise","blur"]

print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("Conditions:", CONDITIONS)

Device: cuda
CUDA available: True
Conditions: ['clean', 'brightness', 'gaussian_noise', 'blur']


In [43]:
import os
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset

test_df = pd.read_csv("../data/metadata/test_split.csv")
images_path = os.path.abspath("../data/raw/images/images")
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class MHISTDataset(Dataset):

    def __init__(self, dataframe, images_path, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images_path = images_path
        self.transform = transform

        # Convert class names into numbers
        self.label_map = {
            "HP": 0,
            "SSA": 1
        }

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        # Get image name and label
        image_name = self.dataframe.loc[index, "Image Name"]
        label_name = self.dataframe.loc[index, "Majority Vote Label"]

        # Create full image path
        image_path = os.path.join(self.images_path, image_name)

        # Open image
        image = Image.open(image_path).convert("RGB")

        # Apply preprocessing/augmentation
        if self.transform:
            image = self.transform(image)

        # Convert label to number
        label = self.label_map[label_name]

        return image, label

print("MHISTDataset class created successfully!")
test_dataset = MHISTDataset(
    test_df,
    images_path,
    transform=val_test_transform
)
print("Test dataset samples:", len(test_dataset))


test_labels = [int(test_dataset[i][1]) for i in range(len(test_dataset))]

print("HP images:", test_labels.count(0))
print("SSA images:", test_labels.count(1))
print("Unique labels:", sorted(set(test_labels)))

MHISTDataset class created successfully!
Test dataset samples: 977
HP images: 617
SSA images: 360
Unique labels: [0, 1]


In [44]:
def create_model(model_name, num_classes=NUM_CLASSES, device=device):
    if model_name == "ResNet18":
        from torchvision.models import resnet18
        model = resnet18(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "EfficientNet-B0":
        from torchvision.models import efficientnet_b0
        model = efficientnet_b0(weights=None)
        model.classifier[1] = torch.nn.Linear(
            model.classifier[1].in_features, num_classes
        )
    elif model_name == "MobileNetV3-Small":
        from torchvision.models import mobilenet_v3_small
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = torch.nn.Linear(
            model.classifier[3].in_features, num_classes
        )
    elif model_name == "ShuffleNetV2-x0.5":
        from torchvision.models import shufflenet_v2_x0_5
        model = shufflenet_v2_x0_5(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Model {model_name} not supported.")
    
    return model.to(device)

In [47]:
from torchvision.models import resnet18
from torchvision.models import efficientnet_b0
from torchvision.models import mobilenet_v3_small
from torchvision.models import shufflenet_v2_x0_5

CHECKPOINT_PATHS = {
    "ResNet18": "../models/checkpoints/resnet18.pth",
    "EfficientNet-B0": "../models/checkpoints/efficientnet_b0.pth",
    "MobileNetV3-Small": "../models/checkpoints/mobilenet_v3_small.pth",
    "ShuffleNetV2-x0.5": "../models/checkpoints/shufflenet_v3_small.pth"
}
def load_checkpoint(model, checkpoint_path):

    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    # Support common checkpoint formats
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        state_dict = checkpoint

    # Handle weights saved from DataParallel
    state_dict = {
        key.removeprefix("module."): value
        for key, value in state_dict.items()
    }

    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    return model
model_names = []
trained_models = {}
missing_checkpoints = []

for name, path in CHECKPOINT_PATHS.items():
    if not os.path.isfile(path):
        missing_checkpoints.append((name, path))
        print(f"Skipping {name}: checkpoint not found at {path}")
        continue

    model = create_model(name, num_classes=NUM_CLASSES, device=device)
    trained_models[name] = load_checkpoint(model, path)
    model_names.append(name)

    print("Loaded:", name)

if missing_checkpoints:
    print("\nTo include the baseline models, run the ResNet18 and EfficientNet-B0 training cells in 03_baseline.ipynb first.")
    print("Those cells save the required checkpoints under ../models/checkpoints/.")

Loaded: ResNet18
Loaded: EfficientNet-B0
Loaded: MobileNetV3-Small
Loaded: ShuffleNetV2-x0.5
